In [31]:
"""
Phase 1: Data Collection
- Download DrugAge C. elegans compounds (positives)
- Fetch SMILES from PubChem (by name first, then batch by CID)
- Save: drugage_raw.csv, drugage_celegans_all.csv, smiles_checkpoint.csv,
        positives_with_smiles.csv

Run from the notebooks/ directory:
    python phase1_fetch_data.py
"""

import io
import time
import zipfile
import requests
import pandas as pd
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

DRUGAGE_URL      = "https://genomics.senescence.info/drugs/dataset.zip"
PUBCHEM_NAME_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{}/property/IsomericSMILES,CanonicalSMILES,MolecularWeight/JSON"
PUBCHEM_CID_URL  = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{}/property/IsomericSMILES,CanonicalSMILES,MolecularWeight/JSON"

SLEEP = 0.2  # PubChem allows ~5 req/sec


# ── Step 1: Download DrugAge ──────────────────────────────────────────────────
def download_drugage() -> pd.DataFrame:
    print("Downloading DrugAge database...")
    resp = requests.get(DRUGAGE_URL, timeout=30)
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        print(f"  Files in zip: {z.namelist()}")
        csv_name = [f for f in z.namelist() if f.endswith(".csv")][0]
        with z.open(csv_name) as f:
            df = pd.read_csv(f, encoding="latin-1")

    # Normalise column names
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    print(f"  Loaded: {df.shape[0]} rows | Columns: {list(df.columns)}")
    return df


# ── Step 2: Filter to C. elegans positives ───────────────────────────────────
def filter_celegans(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    cel = df[df["species"].str.contains("elegans", case=False, na=False)].copy()
    print(f"\nC. elegans entries: {len(cel)}")

    cel["avg_lifespan_change_percent"] = pd.to_numeric(
        cel["avg_lifespan_change_percent"], errors="coerce"
    )
    positives = cel[cel["avg_lifespan_change_percent"] > 0].copy()
    print(f"  Positive (> 0):  {len(positives)}")
    print(f"  Zero/neg (<= 0): {(cel['avg_lifespan_change_percent'] <= 0).sum()}")
    print(f"  NaN:             {cel['avg_lifespan_change_percent'].isna().sum()}")
    return cel, positives


# ── Step 3: Fetch SMILES by name ──────────────────────────────────────────────
def fetch_by_name(name: str) -> dict:
    """Fetch CID + SMILES from PubChem by compound name."""
    url = PUBCHEM_NAME_URL.format(requests.utils.quote(name))
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            p = resp.json()["PropertyTable"]["Properties"][0]
            return {
                "name": name,
                "cid": p.get("CID"),
                "smiles": p.get("SMILES") or p.get("IsomericSMILES") or p.get("CanonicalSMILES"),
                "mol_weight": p.get("MolecularWeight"),
                "found": True,
                "error": None,
            }
        return {"name": name, "found": False, "error": f"HTTP {resp.status_code}",
                "cid": None, "smiles": None, "mol_weight": None}
    except Exception as e:
        return {"name": name, "found": False, "error": str(e),
                "cid": None, "smiles": None, "mol_weight": None}


def fetch_smiles_by_name(names: list, save_path: Path) -> pd.DataFrame:
    """Fetch SMILES for all names with checkpointing."""
    if save_path.exists():
        done_df = pd.read_csv(save_path)
        done_names = set(done_df["name"].tolist())
        results = done_df.to_dict("records")
        print(f"  Resuming from checkpoint: {len(done_names)} already fetched")
    else:
        done_names, results = set(), []

    remaining = [n for n in names if n not in done_names]
    print(f"  Fetching {len(remaining)} remaining compounds...")

    for i, name in enumerate(remaining):
        results.append(fetch_by_name(name))
        if (i + 1) % 50 == 0:
            pd.DataFrame(results).to_csv(save_path, index=False)
            found = sum(r["found"] for r in results)
            print(f"  [{i+1}/{len(remaining)}] Found: {found}/{len(results)}")
        time.sleep(SLEEP)

    df = pd.DataFrame(results)
    df.to_csv(save_path, index=False)
    return df


# ── Step 4: Patch missing SMILES via batch CID fetch ─────────────────────────
def patch_missing_smiles(checkpoint: pd.DataFrame) -> pd.DataFrame:
    """
    For rows that have a CID but no SMILES (can happen if the name endpoint
    returns CID/mol_weight but omits SMILES), re-fetch via the CID endpoint
    in batches of 100.
    """
    needs_patch = checkpoint[
        checkpoint["found"] & checkpoint["cid"].notna() & checkpoint["smiles"].isna()
    ].copy()

    if len(needs_patch) == 0:
        print("  No SMILES patching needed.")
        return checkpoint

    print(f"  Patching {len(needs_patch)} rows missing SMILES via CID batch fetch...")
    cids = needs_patch["cid"].astype(int).tolist()
    cid_to_smiles = {}

    for i in range(0, len(cids), 100):
        batch = cids[i:i+100]
        url = PUBCHEM_CID_URL.format(",".join(str(c) for c in batch))
        try:
            resp = requests.get(url, timeout=15)
            if resp.status_code == 200:
                for p in resp.json()["PropertyTable"]["Properties"]:
                    cid_to_smiles[p["CID"]] = {
                        "smiles": p.get("SMILES") or p.get("IsomericSMILES") or p.get("CanonicalSMILES"),
                        "mol_weight": p.get("MolecularWeight"),
                    }
        except Exception as e:
            print(f"  Batch error: {e}")
        time.sleep(SLEEP)

    # Map back
    for idx, row in needs_patch.iterrows():
        cid = int(row["cid"])
        if cid in cid_to_smiles:
            checkpoint.at[idx, "smiles"]     = cid_to_smiles[cid]["smiles"]
            checkpoint.at[idx, "mol_weight"] = cid_to_smiles[cid]["mol_weight"]

    patched = checkpoint["found"] & checkpoint["smiles"].notna()
    print(f"  After patching — rows with SMILES: {patched.sum()}")
    return checkpoint


# ── Step 5: Build positives dataset ──────────────────────────────────────────
def build_positives(positives: pd.DataFrame, checkpoint: pd.DataFrame) -> pd.DataFrame:
    """Merge DrugAge positives with their SMILES from checkpoint."""
    ckpt = checkpoint[checkpoint["found"] & checkpoint["smiles"].notna()][
        ["name", "cid", "smiles", "mol_weight"]
    ].rename(columns={"name": "compound_name"})

    merged = positives.merge(ckpt, on="compound_name", how="inner")
    merged["label"] = 1
    print(f"\nPositives with SMILES: {len(merged)} / {len(positives)}")
    print(f"  ({positives['compound_name'].nunique()} unique compounds, "
          f"{ckpt['compound_name'].nunique()} matched)")
    return merged


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # 1. Download DrugAge
    raw_df = download_drugage()
    raw_df.to_csv(OUTPUT_DIR / "drugage_raw.csv", index=False)

    # 2. Filter to C. elegans
    cel_df, positives = filter_celegans(raw_df)
    cel_df.to_csv(OUTPUT_DIR / "drugage_celegans_all.csv", index=False)

    # 3. Fetch SMILES by name
    names = positives["compound_name"].dropna().unique().tolist()
    print(f"\nFetching SMILES for {len(names)} unique positive compounds...")
    checkpoint = fetch_smiles_by_name(names, OUTPUT_DIR / "smiles_checkpoint.csv")

    # 4. Patch any rows with CID but missing SMILES
    print("\nPatching missing SMILES via CID batch fetch...")
    checkpoint = patch_missing_smiles(checkpoint)
    checkpoint.to_csv(OUTPUT_DIR / "smiles_checkpoint.csv", index=False)

    found     = checkpoint[checkpoint["found"] & checkpoint["smiles"].notna()]
    not_found = checkpoint[~checkpoint["found"] | checkpoint["smiles"].isna()]
    print(f"\nFinal checkpoint: {len(found)} with SMILES, {len(not_found)} without")
    not_found.to_csv(OUTPUT_DIR / "smiles_not_found.csv", index=False)

    # 5. Build positives dataset
    positives_out = build_positives(positives, checkpoint)
    positives_out.to_csv(OUTPUT_DIR / "positives_with_smiles.csv", index=False)

    print(f"\n✅ Phase 1 complete!")
    print(f"   Saved: positives_with_smiles.csv ({len(positives_out)} rows)")
    print(f"   Next:  python phase1b_build_negatives.py")


  Files in zip: ['drugage.csv', 'release.html']
  Loaded: 3423 rows | Columns: ['compound_name', 'species', 'strain', 'dosage', 'age_at_initiation', 'treatment_duration', 'avg_lifespan_change_percent', 'avg_lifespan_significance', 'max_lifespan_change_percent', 'max_lifespan_significance', 'gender', 'weight_change_percent', 'weight_change_significance', 'itp', 'pubmed_id']

C. elegans entries: 1586
  Positive (> 0):  1209
  Zero/neg (<= 0): 362
  NaN:             15

Fetching SMILES for 556 unique positive compounds...
  Resuming from checkpoint: 556 already fetched
  Fetching 0 remaining compounds...

Patching missing SMILES via CID batch fetch...
  No SMILES patching needed.

Final checkpoint: 430 with SMILES, 126 without

Positives with SMILES: 995 / 1209
  (556 unique compounds, 430 matched)

✅ Phase 1 complete!
   Saved: positives_with_smiles.csv (995 rows)
   Next:  python phase1b_build_negatives.py


In [32]:
"""
Phase 1b: Build Negative Class + Assemble Final Labeled Dataset

Negative compounds come from two sources:
  1. DrugAge C. elegans entries with avg_lifespan_change_percent <= 0
     (explicitly tested, no lifespan extension)
  2. ChEMBL approved small molecules (Phase 4) not present in DrugAge positives
     (approved drugs with no known C. elegans lifespan data)

This gives a neg:pos ratio of ~5:1, matching the paper's class distribution.
Class imbalance is also handled downstream with class_weight='balanced' in the RF.

Requires (from Phase 1):
  - data/drugage_celegans_all.csv
  - data/positives_with_smiles.csv
  - data/smiles_checkpoint.csv

Outputs:
  - data/labeled_dataset.csv   ← final dataset for Phase 2 onwards

Run from the notebooks/ directory:
    python phase1b_build_negatives.py
"""

import requests
import time
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("data")

PUBCHEM_NAME_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{}/property/IsomericSMILES,CanonicalSMILES,MolecularWeight/JSON"
CHEMBL_URL       = "https://www.ebi.ac.uk/chembl/api/data/molecule?format=json&molecule_type=Small+molecule&max_phase=4&limit=1000&offset={offset}"
SLEEP = 0.2


# ── Step 1: Load Phase 1 outputs ──────────────────────────────────────────────
def load_phase1():
    raw        = pd.read_csv(OUTPUT_DIR / "drugage_celegans_all.csv")
    positives  = pd.read_csv(OUTPUT_DIR / "positives_with_smiles.csv")
    checkpoint = pd.read_csv(OUTPUT_DIR / "smiles_checkpoint.csv")

    print("=== Phase 1 outputs ===")
    print(f"C. elegans entries:        {len(raw)}")
    print(f"Positive rows:             {len(positives)}")
    print(f"  smiles non-null:         {positives['smiles'].notna().sum()}")
    print(f"Checkpoint rows:           {len(checkpoint)}")
    print(f"  with SMILES:             {checkpoint['smiles'].notna().sum()}")
    return raw, positives, checkpoint


# ── Step 2a: DrugAge negatives ────────────────────────────────────────────────
def get_drugage_negatives(raw: pd.DataFrame, checkpoint: pd.DataFrame) -> pd.DataFrame:
    """
    Compounds explicitly tested in C. elegans with no lifespan increase.
    Fetch SMILES from PubChem, using the existing checkpoint as cache.
    """
    neg_entries = raw[raw["avg_lifespan_change_percent"] <= 0].copy()
    neg_names   = neg_entries["compound_name"].dropna().unique().tolist()

    print(f"\n=== Source 1: DrugAge negatives ===")
    print(f"Entries with effect <= 0:  {len(neg_entries)}")
    print(f"Unique compound names:     {len(neg_names)}")

    # Build cache from Phase 1 checkpoint
    cache = {r["name"]: r for r in checkpoint.to_dict("records")
             if pd.notna(r.get("name"))}

    save_path = OUTPUT_DIR / "neg_drugage_smiles.csv"
    if save_path.exists():
        existing = pd.read_csv(save_path)
        for r in existing.to_dict("records"):
            if pd.notna(r.get("name")):
                cache[r["name"]] = r
        print(f"  Loaded {len(existing)} rows from checkpoint")

    results, to_fetch = [], []
    for name in neg_names:
        if name in cache:
            results.append(cache[name])
        else:
            to_fetch.append(name)

    print(f"  From cache: {len(results)} | To fetch: {len(to_fetch)}")

    for i, name in enumerate(to_fetch):
        url = PUBCHEM_NAME_URL.format(requests.utils.quote(name))
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                p = resp.json()["PropertyTable"]["Properties"][0]
                smiles = p.get("SMILES") or p.get("IsomericSMILES") or p.get("CanonicalSMILES")
                results.append({"name": name, "cid": p.get("CID"), "smiles": smiles,
                                 "mol_weight": p.get("MolecularWeight"), "found": True})
            else:
                results.append({"name": name, "found": False, "smiles": None})
        except Exception:
            results.append({"name": name, "found": False, "smiles": None})
        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{len(to_fetch)}]")
        time.sleep(SLEEP)

    df = pd.DataFrame(results)
    df.to_csv(save_path, index=False)

    # Build negatives dataframe
    neg = df[df["smiles"].notna()][["name", "smiles", "mol_weight"]].copy()
    neg = neg.rename(columns={"name": "compound_name"})
    neg["label"] = 0
    print(f"  DrugAge negatives with SMILES: {len(neg)}")
    return neg


# ── Step 2b: ChEMBL negatives ─────────────────────────────────────────────────
def get_chembl_negatives(positives: pd.DataFrame,
                         target: int = 2000) -> pd.DataFrame:
    """
    Approved small molecules from ChEMBL (Phase 4) not in DrugAge positives.
    ChEMBL provides SMILES directly — no PubChem lookup needed.
    """
    print(f"\n=== Source 2: ChEMBL negatives (target: {target}) ===")

    save_path = OUTPUT_DIR / "chembl_raw.csv"
    if save_path.exists():
        print("  Loading cached ChEMBL data...")
        df = pd.read_csv(save_path)
    else:
        print("  Downloading from ChEMBL API...")
        records = []
        offset  = 0
        while True:
            url  = CHEMBL_URL.format(offset=offset)
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            data      = resp.json()
            molecules = data.get("molecules", [])
            if not molecules:
                break
            for m in molecules:
                name   = m.get("pref_name") or m.get("chembl_id")
                struct = m.get("molecule_structures") or {}
                props  = m.get("molecule_properties") or {}
                records.append({
                    "compound_name": name,
                    "smiles":        struct.get("canonical_smiles"),
                    "mol_weight":    props.get("full_mwt"),
                })
            offset += len(molecules)
            print(f"  Fetched {len(records)} so far...")
            if len(molecules) < 1000:
                break
            time.sleep(SLEEP)

        df = pd.DataFrame(records)
        df.to_csv(save_path, index=False)
        print(f"  Total: {len(df)} | With SMILES: {df['smiles'].notna().sum()}")

    # Exclude known positives by name and SMILES
    positive_names  = set(positives["compound_name"].dropna().str.lower())
    positive_smiles = set(positives["smiles"].dropna())

    candidates = df[
        df["smiles"].notna() &
        (~df["compound_name"].str.lower().isin(positive_names)) &
        (~df["smiles"].isin(positive_smiles))
    ].copy()

    # Filter >3000 Da (paper excluded these)
    candidates["mol_weight"] = pd.to_numeric(candidates["mol_weight"], errors="coerce")
    candidates = candidates[candidates["mol_weight"] <= 3000]

    # Sample down to target
    if len(candidates) > target:
        candidates = candidates.sample(n=target, random_state=42)

    candidates["label"] = 0
    print(f"  ChEMBL negatives after filtering: {len(candidates)}")
    return candidates[["compound_name", "smiles", "mol_weight", "label"]]


# ── Step 3: Assemble final dataset ────────────────────────────────────────────
def build_final_dataset(positives: pd.DataFrame,
                        drugage_neg: pd.DataFrame,
                        chembl_neg: pd.DataFrame) -> pd.DataFrame:
    pos = positives[["compound_name", "smiles", "mol_weight"]].copy()
    pos["label"] = 1

    print(f"\n=== Assembling final dataset ===")
    print(f"Positives:         {len(pos)}")
    print(f"DrugAge negatives: {len(drugage_neg)}")
    print(f"ChEMBL negatives:  {len(chembl_neg)}")

    # Combine — positives first so they win on any conflict
    combined = pd.concat([pos, drugage_neg, chembl_neg], ignore_index=True)
    combined = combined.dropna(subset=["smiles"])
    combined = combined.sort_values("label", ascending=False)
    combined = combined.drop_duplicates(subset=["smiles"],        keep="first")
    combined = combined.drop_duplicates(subset=["compound_name"], keep="first")
    combined = combined.reset_index(drop=True)

    print(f"\nAfter dedup:")
    print(f"  Positive: {(combined['label'] == 1).sum()}")
    print(f"  Negative: {(combined['label'] == 0).sum()}")
    print(f"  Total:    {len(combined)}")
    ratio = (combined['label'] == 0).sum() / max((combined['label'] == 1).sum(), 1)
    print(f"  Neg:Pos ratio: {ratio:.1f}:1")
    return combined


# ── Summary ───────────────────────────────────────────────────────────────────
def print_summary(df: pd.DataFrame):
    print("\n" + "="*50)
    print("FINAL DATASET SUMMARY")
    print("="*50)
    print(f"Total:      {len(df)}")
    print(f"Positive:   {(df['label']==1).sum()}")
    print(f"Negative:   {(df['label']==0).sum()}")
    print(f"% Positive: {(df['label']==1).mean()*100:.1f}%")
    ratio = (df['label']==0).sum() / max((df['label']==1).sum(), 1)
    print(f"Neg:Pos:    {ratio:.1f}:1")
    print(f"\nPaper: 229 pos / 1163 neg / 1392 total (~5:1)")
    print(f"Ours:  {(df['label']==1).sum()} pos / {(df['label']==0).sum()} neg / {len(df)} total")
    df["mol_weight"] = pd.to_numeric(df["mol_weight"], errors="coerce")
    print(f"\nMol weight — median: {df['mol_weight'].median():.0f} Da")
    print(f"  >1000 Da: {(df['mol_weight'] > 1000).sum()}")
    print(f"  >3000 Da: {(df['mol_weight'] > 3000).sum()}")
    print("="*50)


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    raw, positives, checkpoint = load_phase1()

    drugage_neg = get_drugage_negatives(raw, checkpoint)
    chembl_neg  = get_chembl_negatives(positives, target=2000)

    final = build_final_dataset(positives, drugage_neg, chembl_neg)
    print_summary(final)

    final.to_csv(OUTPUT_DIR / "labeled_dataset.csv", index=False)
    print(f"\n✅ Saved: data/labeled_dataset.csv")
    print("Next: python phase2a_rdkit_descriptors.py")


=== Phase 1 outputs ===
C. elegans entries:        1586
Positive rows:             995
  smiles non-null:         995
Checkpoint rows:           556
  with SMILES:             430

=== Source 1: DrugAge negatives ===
Entries with effect <= 0:  362
Unique compound names:     230
  Loaded 230 rows from checkpoint
  From cache: 230 | To fetch: 0
  DrugAge negatives with SMILES: 178

=== Source 2: ChEMBL negatives (target: 2000) ===
  Loading cached ChEMBL data...
  ChEMBL negatives after filtering: 2000

=== Assembling final dataset ===
Positives:         995
DrugAge negatives: 178
ChEMBL negatives:  2000

After dedup:
  Positive: 428
  Negative: 2093
  Total:    2521
  Neg:Pos ratio: 4.9:1

FINAL DATASET SUMMARY
Total:      2521
Positive:   428
Negative:   2093
% Positive: 17.0%
Neg:Pos:    4.9:1

Paper: 229 pos / 1163 neg / 1392 total (~5:1)
Ours:  428 pos / 2093 neg / 2521 total

Mol weight — median: 361 Da
  >1000 Da: 53
  >3000 Da: 0

✅ Saved: data/labeled_dataset.csv
Next: python ph

In [33]:
"""
Phase 2a: Compute RDKit Chemical Descriptors

The paper used MOE (commercial) to compute 268 2D/3D molecular descriptors.
We use RDKit (free) to compute an equivalent set covering the same categories:
  - Electronic:   partial charges, polar surface area, electron distribution
  - Steric:       molecular size, shape, connectivity indices
  - Hydrophobic:  logP, hydrophobic surface area

RDKit provides ~200 2D descriptors. We compute all of them, then apply the
same filtering the paper used:
  - Remove descriptors with >98% constant values (near-zero variance)
  - Impute missing values (MissForest equivalent: median imputation per column)

Requires:
  - data/labeled_dataset.csv
  - pip install rdkit

Outputs:
  - data/chemical_descriptors.csv   (one row per compound, ~200 descriptor cols)
  - data/dataset_with_descriptors.csv  (labeled_dataset + descriptors merged)

Run from notebooks/ directory:
    python phase2a_rdkit_descriptors.py
"""

import numpy as np
import pandas as pd
from pathlib import Path

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, QED
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Descriptors import MoleculeDescriptors

OUTPUT_DIR = Path("data")

# Threshold for removing near-constant descriptors (matching paper's 98%)
CONSTANT_THRESHOLD = 0.98


# ── Step 1: Load dataset ──────────────────────────────────────────────────────
def load_dataset() -> pd.DataFrame:
    df = pd.read_csv(OUTPUT_DIR / "labeled_dataset.csv")
    print(f"Loaded: {len(df)} compounds | "
          f"{(df['label']==1).sum()} pos / {(df['label']==0).sum()} neg")
    print(f"SMILES non-null: {df['smiles'].notna().sum()}")
    return df


# ── Step 2: Compute RDKit descriptors ────────────────────────────────────────
def get_descriptor_names() -> list:
    """Return all RDKit 2D descriptor names."""
    return [d[0] for d in Descriptors.descList]


def compute_descriptors_for_mol(mol) -> dict:
    """Compute all RDKit descriptors for a single molecule."""
    desc_names = get_descriptor_names()
    calc = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)
    try:
        values = calc.CalcDescriptors(mol)
        return dict(zip(desc_names, values))
    except Exception:
        return {name: np.nan for name in desc_names}


def compute_all_descriptors(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute RDKit descriptors for all compounds.
    Invalid SMILES are skipped (descriptors set to NaN).
    """
    desc_names = get_descriptor_names()
    print(f"\nComputing {len(desc_names)} RDKit descriptors for {len(df)} compounds...")

    records = []
    failed  = 0

    for i, (_, row) in enumerate(df.iterrows()):
        smiles = row["smiles"]
        try:
            mol = Chem.MolFromSmiles(str(smiles))
            if mol is None:
                raise ValueError("Invalid SMILES")
            desc = compute_descriptors_for_mol(mol)
            desc["compound_name"] = row["compound_name"]
            desc["smiles"]        = smiles
            desc["label"]         = row["label"]
        except Exception:
            failed += 1
            desc = {name: np.nan for name in desc_names}
            desc["compound_name"] = row["compound_name"]
            desc["smiles"]        = smiles
            desc["label"]         = row["label"]

        records.append(desc)

        if (i + 1) % 250 == 0:
            print(f"  [{i+1}/{len(df)}] Failed so far: {failed}")

    df_desc = pd.DataFrame(records)
    print(f"\nDone. Failed/invalid SMILES: {failed} / {len(df)}")
    return df_desc


# ── Step 3: Filter near-constant descriptors ──────────────────────────────────
def filter_constant_descriptors(df_desc: pd.DataFrame,
                                 threshold: float = CONSTANT_THRESHOLD) -> pd.DataFrame:
    """
    Remove descriptors where >= threshold fraction of values are identical.
    Matches the paper's '>98% constant values' filter.
    """
    desc_cols = [c for c in df_desc.columns
                 if c not in ("compound_name", "smiles", "label")]

    to_drop = []
    for col in desc_cols:
        top_freq = df_desc[col].value_counts(normalize=True, dropna=False).iloc[0]
        if top_freq >= threshold:
            to_drop.append(col)

    print(f"\nDescriptor filtering:")
    print(f"  Before: {len(desc_cols)} descriptors")
    print(f"  Dropping {len(to_drop)} near-constant descriptors (>={threshold*100:.0f}% same value)")
    print(f"  After:  {len(desc_cols) - len(to_drop)} descriptors")

    return df_desc.drop(columns=to_drop)


# ── Step 4: Impute missing values ─────────────────────────────────────────────
def impute_missing(df_desc: pd.DataFrame) -> pd.DataFrame:
    """
    Impute missing descriptor values using column median.
    The paper used MissForest (RF-based imputation); median is a simpler
    but effective approximation that avoids circular dependency issues.
    """
    desc_cols = [c for c in df_desc.columns
                 if c not in ("compound_name", "smiles", "label")]

    missing_before = df_desc[desc_cols].isna().sum().sum()
    print(f"\nMissing values before imputation: {missing_before}")

    # Replace inf values with NaN first
    df_desc[desc_cols] = df_desc[desc_cols].replace([np.inf, -np.inf], np.nan)

    # Median imputation per column
    for col in desc_cols:
        median = df_desc[col].median()
        df_desc[col] = df_desc[col].fillna(median)

    missing_after = df_desc[desc_cols].isna().sum().sum()
    print(f"Missing values after imputation:  {missing_after}")
    return df_desc


# ── Step 5: Summary ───────────────────────────────────────────────────────────
def print_summary(df_desc: pd.DataFrame):
    desc_cols = [c for c in df_desc.columns
                 if c not in ("compound_name", "smiles", "label")]
    print("\n" + "="*50)
    print("CHEMICAL DESCRIPTORS SUMMARY")
    print("="*50)
    print(f"Compounds:         {len(df_desc)}")
    print(f"Descriptors:       {len(desc_cols)}")
    print(f"  Paper used:      268 (MOE)")
    print(f"  Ours (RDKit):    {len(desc_cols)}")
    print(f"Positive:          {(df_desc['label']==1).sum()}")
    print(f"Negative:          {(df_desc['label']==0).sum()}")
    print(f"\nSample descriptors: {desc_cols[:5]}")
    print("="*50)


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # 1. Load
    df = load_dataset()

    # 2. Compute descriptors
    df_desc = compute_all_descriptors(df)

    # 3. Filter near-constant
    df_desc = filter_constant_descriptors(df_desc)

    # 4. Impute missing
    df_desc = impute_missing(df_desc)

    # 5. Summary
    print_summary(df_desc)

    # 6. Save
    desc_cols = [c for c in df_desc.columns
                 if c not in ("compound_name", "smiles", "label")]

    # Save descriptors only (for merging with GO terms later)
    df_desc[["compound_name", "smiles", "label"] + desc_cols].to_csv(
        OUTPUT_DIR / "chemical_descriptors.csv", index=False
    )
    print(f"\n✅ Saved: data/chemical_descriptors.csv")
    print(f"   Shape: {df_desc.shape}")
    print("Next: python phase2b_go_terms.py")

Loaded: 2521 compounds | 428 pos / 2093 neg
SMILES non-null: 2521

Computing 217 RDKit descriptors for 2521 compounds...
  [250/2521] Failed so far: 0
  [500/2521] Failed so far: 0
  [750/2521] Failed so far: 0
  [1000/2521] Failed so far: 0
  [1250/2521] Failed so far: 0


[23:13:56] WARNING: not removing hydrogen atom without neighbors
[23:13:56] WARNING: not removing hydrogen atom without neighbors
[23:13:56] WARNING: not removing hydrogen atom without neighbors
[23:13:56] WARNING: not removing hydrogen atom without neighbors
[23:13:57] WARNING: not removing hydrogen atom without neighbors
[23:13:57] WARNING: not removing hydrogen atom without neighbors
[23:13:57] WARNING: not removing hydrogen atom without neighbors
[23:13:57] WARNING: not removing hydrogen atom without neighbors


  [1500/2521] Failed so far: 0
  [1750/2521] Failed so far: 0
  [2000/2521] Failed so far: 0
  [2250/2521] Failed so far: 0
  [2500/2521] Failed so far: 0

Done. Failed/invalid SMILES: 0 / 2521

Descriptor filtering:
  Before: 217 descriptors
  Dropping 38 near-constant descriptors (>=98% same value)
  After:  179 descriptors

Missing values before imputation: 1720
Missing values after imputation:  0

CHEMICAL DESCRIPTORS SUMMARY
Compounds:         2521
Descriptors:       179
  Paper used:      268 (MOE)
  Ours (RDKit):    179
Positive:          428
Negative:          2093

Sample descriptors: ['MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed']

✅ Saved: data/chemical_descriptors.csv
   Shape: (2521, 182)
Next: python phase2b_go_terms.py


In [35]:
import os
os.remove("data/compound_targets.csv")

In [36]:
"""
Phase 2b: Biological GO Term Features via ChEMBL

Pipeline:
  1. For each compound, look up its ChEMBL ID by name
  2. Fetch mechanism of action → target_chembl_id(s)
  3. Fetch target details → GO terms from target_component_xrefs
  4. Build binary GO term feature matrix (one column per GO term)

This replaces the paper's STITCH + ClueGO approach, which used a now-defunct
API. ChEMBL provides the same drug→target→GO term chain in a single API.

Note: ChEMBL has curated targets primarily for approved/investigational drugs.
Coverage will be highest for DrugAge positives and ChEMBL-sourced negatives,
and lower for less-studied compounds. Compounds with no targets get all-zero
GO feature vectors — this is fine for the RF model.

Requires:
  - data/labeled_dataset.csv
  - data/chembl_raw.csv  (from Phase 1b — contains ChEMBL IDs)

Outputs:
  - data/compound_chembl_ids.csv    (compound → ChEMBL ID)
  - data/compound_targets.csv       (compound → target ChEMBL IDs)
  - data/target_go_terms.csv        (target → GO terms)
  - data/go_term_features.csv       (binary GO term matrix)

Run from notebooks/ directory:
    python phase2b_go_terms.py
"""

import requests
import time
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

OUTPUT_DIR = Path("data")

CHEMBL_MOL_URL    = "https://www.ebi.ac.uk/chembl/api/data/molecule?pref_name__iexact={name}&format=json&limit=1"
CHEMBL_MECH_URL   = "https://www.ebi.ac.uk/chembl/api/data/mechanism?molecule_chembl_id={cid}&format=json"
CHEMBL_TARGET_URL = "https://www.ebi.ac.uk/chembl/api/data/target/{tid}?format=json"

SLEEP = 0.2
CONSTANT_THRESHOLD = 0.98


# ── Step 1: Resolve compound names → ChEMBL IDs ───────────────────────────────
def resolve_chembl_ids(df: pd.DataFrame, chembl_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Use ChEMBL IDs already in chembl_raw where possible (fast),
    fall back to API name lookup for DrugAge compounds not in ChEMBL.
    """
    save_path = OUTPUT_DIR / "compound_chembl_ids.csv"
    if save_path.exists():
        print("  Loading cached ChEMBL ID mappings...")
        return pd.read_csv(save_path)

    # Build lookup from chembl_raw (already has compound_name → ChEMBL data)
    # ChEMBL IDs are in chembl_raw if we fetched them in Phase 1b
    # For now we need to fetch them via the API for all compounds
    compounds = df["compound_name"].dropna().unique().tolist()
    print(f"  Resolving ChEMBL IDs for {len(compounds)} compounds...")

    results = []
    for i, name in enumerate(compounds):
        url = CHEMBL_MOL_URL.format(name=requests.utils.quote(str(name)))
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                mols = resp.json().get("molecules", [])
                if mols:
                    results.append({
                        "compound_name": name,
                        "chembl_id": mols[0]["molecule_chembl_id"],
                        "found": True,
                    })
                else:
                    results.append({"compound_name": name, "chembl_id": None, "found": False})
            else:
                results.append({"compound_name": name, "chembl_id": None, "found": False})
        except Exception:
            results.append({"compound_name": name, "chembl_id": None, "found": False})

        if (i + 1) % 100 == 0:
            found = sum(r["found"] for r in results)
            print(f"  [{i+1}/{len(compounds)}] Found: {found}/{len(results)}")
        time.sleep(SLEEP)

    id_df = pd.DataFrame(results)
    id_df.to_csv(save_path, index=False)
    found = id_df["found"].sum()
    print(f"  ChEMBL IDs resolved: {found} / {len(compounds)}")
    return id_df


# ── Step 2: Fetch mechanism → target ChEMBL IDs ───────────────────────────────
def fetch_targets(id_df: pd.DataFrame) -> pd.DataFrame:
    """For each compound with a ChEMBL ID, fetch its protein targets."""
    save_path = OUTPUT_DIR / "compound_targets.csv"
    if save_path.exists():
        print("  Loading cached compound targets...")
        return pd.read_csv(save_path)

    has_id = id_df[id_df["chembl_id"].notna()]
    print(f"  Fetching targets for {len(has_id)} compounds with ChEMBL IDs...")

    records = []
    for i, (_, row) in enumerate(has_id.iterrows()):
        url = CHEMBL_MECH_URL.format(cid=row["chembl_id"])
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                mechs = resp.json().get("mechanisms", [])
                for m in mechs:
                    tid = m.get("target_chembl_id")
                    if tid:
                        records.append({
                            "compound_name": row["compound_name"],
                            "chembl_id":     row["chembl_id"],
                            "target_id":     tid,
                            "action_type":   m.get("action_type"),
                        })
                if not mechs:
                    records.append({
                        "compound_name": row["compound_name"],
                        "chembl_id":     row["chembl_id"],
                        "target_id":     None,
                        "action_type":   None,
                    })
        except Exception:
            records.append({
                "compound_name": row["compound_name"],
                "chembl_id":     row["chembl_id"],
                "target_id":     None,
                "action_type":   None,
            })

        if (i + 1) % 100 == 0:
            has_targets = sum(1 for r in records if r["target_id"])
            print(f"  [{i+1}/{len(has_id)}] With targets: {has_targets}")
        time.sleep(SLEEP)

    targets_df = pd.DataFrame(records)
    targets_df.to_csv(save_path, index=False)
    with_targets = targets_df[targets_df["target_id"].notna()]["compound_name"].nunique()
    print(f"  Compounds with ≥1 target: {with_targets} / {len(has_id)}")
    return targets_df


# ── Step 3: Fetch GO terms for each target ────────────────────────────────────
def fetch_go_terms(targets_df: pd.DataFrame) -> pd.DataFrame:
    """Fetch GO terms for each unique target ChEMBL ID."""
    save_path = OUTPUT_DIR / "target_go_terms.csv"
    if save_path.exists():
        print("  Loading cached target GO terms...")
        return pd.read_csv(save_path)

    unique_targets = targets_df[targets_df["target_id"].notna()]["target_id"].unique().tolist()
    print(f"  Fetching GO terms for {len(unique_targets)} unique targets...")

    records = []
    for i, tid in enumerate(unique_targets):
        url = CHEMBL_TARGET_URL.format(tid=tid)
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                for comp in data.get("target_components", []):
                    for xref in comp.get("target_component_xrefs", []):
                        src = xref.get("xref_src_db", "")
                        if src.startswith("Go"):
                            records.append({
                                "target_id": tid,
                                "go_id":     xref.get("xref_id"),
                                "go_name":   xref.get("xref_name"),
                                "go_aspect": src,  # GoComponent, GoFunction, GoProcess
                            })
        except Exception:
            pass

        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{len(unique_targets)}] GO records so far: {len(records)}")
        time.sleep(SLEEP)

    go_df = pd.DataFrame(records) if records else pd.DataFrame(
        columns=["target_id", "go_id", "go_name", "go_aspect"]
    )
    go_df.to_csv(save_path, index=False)
    print(f"  Total GO term records: {len(go_df)}")
    print(f"  Unique GO terms: {go_df['go_name'].nunique()}")
    return go_df


# ── Step 4: Build binary GO term feature matrix ───────────────────────────────
def build_go_matrix(df: pd.DataFrame,
                    targets_df: pd.DataFrame,
                    go_df: pd.DataFrame) -> pd.DataFrame:
    print("\nBuilding binary GO term matrix...")

    # target_id → set of GO names
    target_to_go = defaultdict(set)
    for _, row in go_df[go_df["go_name"].notna()].iterrows():
        target_to_go[row["target_id"]].add(row["go_name"])

    # compound → set of GO names (via its targets)
    compound_to_go = defaultdict(set)
    for _, row in targets_df[targets_df["target_id"].notna()].iterrows():
        compound_to_go[row["compound_name"]].update(
            target_to_go.get(row["target_id"], set())
        )

    all_go = sorted(set(g for gs in compound_to_go.values() for g in gs))
    compounds = df["compound_name"].tolist()
    has_go = sum(1 for c in compounds if compound_to_go[c])
    print(f"  Unique GO terms (before filter): {len(all_go)}")
    print(f"  Compounds with ≥1 GO term: {has_go} / {len(compounds)}")

    # Build matrix
    rows = []
    for name in compounds:
        cgo = compound_to_go[name]
        row = {go: (1 if go in cgo else 0) for go in all_go}
        row["compound_name"] = name
        rows.append(row)

    matrix = pd.DataFrame(rows)

    # Filter near-constant GO terms (≥98% same value)
    go_cols = [c for c in matrix.columns if c != "compound_name"]
    to_drop = [c for c in go_cols
               if matrix[c].value_counts(normalize=True).iloc[0] >= CONSTANT_THRESHOLD]
    matrix = matrix.drop(columns=to_drop)
    remaining = [c for c in matrix.columns if c != "compound_name"]
    print(f"  After 98% constant filter: {len(remaining)} GO terms")
    print(f"  (Dropped {len(to_drop)} near-constant terms)")

    # Add label
    label_map = df.set_index("compound_name")["label"]
    matrix["label"] = matrix["compound_name"].map(label_map)

    return matrix


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = pd.read_csv(OUTPUT_DIR / "labeled_dataset.csv")
    chembl_raw = pd.read_csv(OUTPUT_DIR / "chembl_raw.csv")
    print(f"Loaded {len(df)} compounds")

    # Step 1: Resolve ChEMBL IDs
    print("\n--- Step 1: Resolve ChEMBL IDs ---")
    id_df = resolve_chembl_ids(df, chembl_raw)

    # Step 2: Fetch targets
    print("\n--- Step 2: Fetch compound targets ---")
    targets_df = fetch_targets(id_df)

    # Step 3: Fetch GO terms
    print("\n--- Step 3: Fetch target GO terms ---")
    go_df = fetch_go_terms(targets_df)

    # Step 4: Build matrix
    print("\n--- Step 4: Build GO term feature matrix ---")
    matrix = build_go_matrix(df, targets_df, go_df)

    matrix.to_csv(OUTPUT_DIR / "go_term_features.csv", index=False)
    go_cols = [c for c in matrix.columns if c not in ("compound_name", "label")]

    print(f"\n{'='*50}")
    print("GO TERM FEATURES SUMMARY")
    print(f"{'='*50}")
    print(f"Compounds:       {len(matrix)}")
    print(f"GO term features:{len(go_cols)}")
    print(f"Paper had:       10,757 GO terms (STITCH-based, now defunct)")
    print(f"{'='*50}")
    print(f"\n✅ Saved: data/go_term_features.csv")
    print("Next: python phase3_model.py")

Loaded 2521 compounds

--- Step 1: Resolve ChEMBL IDs ---
  Loading cached ChEMBL ID mappings...

--- Step 2: Fetch compound targets ---
  Fetching targets for 2306 compounds with ChEMBL IDs...
  [100/2306] With targets: 55
  [200/2306] With targets: 64
  [300/2306] With targets: 120
  [400/2306] With targets: 207
  [500/2306] With targets: 287
  [600/2306] With targets: 371
  [700/2306] With targets: 475
  [800/2306] With targets: 578
  [900/2306] With targets: 658
  [1000/2306] With targets: 739
  [1100/2306] With targets: 815
  [1200/2306] With targets: 892
  [1300/2306] With targets: 994
  [1400/2306] With targets: 1076
  [1500/2306] With targets: 1163
  [1600/2306] With targets: 1249
  [1700/2306] With targets: 1328
  [1800/2306] With targets: 1413
  [1900/2306] With targets: 1488
  [2000/2306] With targets: 1567
  [2100/2306] With targets: 1647
  [2200/2306] With targets: 1724
  [2300/2306] With targets: 1815
  Compounds with ≥1 target: 1357 / 2306

--- Step 3: Fetch target GO te

In [38]:
"""
Phase 3: Random Forest Classifier with Nested Cross-Validation

Replicates Barardo et al. 2017:
  - Nested 10-fold CV (outer loop: performance, inner loop: hyperparameter tuning)
  - Three feature sets evaluated separately and combined:
      1. Chemical descriptors only       (179 RDKit features)
      2. GO term features only           (380 ChEMBL GO terms)
      3. Chemical + GO combined          (559 features)
  - Boruta feature selection (identifies significant features)
  - Metrics: AUC-ROC, accuracy, sensitivity, specificity, MCC
  - class_weight='balanced' to handle 5:1 class imbalance

Paper's best results:
  Chemical only:  AUC 0.730
  GO only:        AUC 0.740
  Combined:       AUC 0.800  ← best

Requires:
  - data/chemical_descriptors.csv
  - data/go_term_features.csv

Outputs:
  - data/results_summary.csv
  - data/feature_importance.csv
  - data/boruta_selected_features.csv
  - plots/roc_curves.png
  - plots/feature_importance.png

Run from notebooks/ directory:
    python phase3_model.py
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix,
    matthews_corrcoef, roc_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

OUTPUT_DIR = Path("data")
PLOTS_DIR  = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
N_FOLDS      = 10

# RF hyperparameters — paper used ranger defaults with class balancing
RF_PARAMS = {
    "n_estimators":  500,
    "max_features":  "sqrt",
    "class_weight":  "balanced",
    "random_state":  RANDOM_STATE,
    "n_jobs":        -1,
}


# ── Step 1: Load and merge feature sets ───────────────────────────────────────
def load_features():
    chem = pd.read_csv(OUTPUT_DIR / "chemical_descriptors.csv")
    go   = pd.read_csv(OUTPUT_DIR / "go_term_features.csv")

    chem_cols = [c for c in chem.columns if c not in ("compound_name", "smiles", "label")]
    go_cols   = [c for c in go.columns   if c not in ("compound_name", "label")]

    print(f"Chemical descriptors: {len(chem_cols)} features, {len(chem)} compounds")
    print(f"GO term features:     {len(go_cols)} features, {len(go)} compounds")

    # Merge on compound_name — inner join keeps only compounds in both
    merged = chem[["compound_name", "label"] + chem_cols].merge(
        go[["compound_name"] + go_cols],
        on="compound_name",
        how="inner"
    )
    print(f"After merge:          {len(merged)} compounds")
    print(f"  Positive: {(merged['label']==1).sum()} | Negative: {(merged['label']==0).sum()}")

    X_chem     = merged[chem_cols].values.astype(float)
    X_go       = merged[go_cols].values.astype(float)
    X_combined = np.hstack([X_chem, X_go])
    y          = merged["label"].values
    names      = merged["compound_name"].values

    return {
        "chemical": (X_chem,     chem_cols, y, names),
        "go":       (X_go,       go_cols,   y, names),
        "combined": (X_combined, chem_cols + go_cols, y, names),
    }, merged


# ── Step 2: Nested CV evaluation ──────────────────────────────────────────────
def evaluate_feature_set(X: np.ndarray, y: np.ndarray,
                          name: str) -> dict:
    """
    Nested 10-fold CV:
      - Outer loop: evaluate model performance
      - Inner loop: not needed for RF with fixed params (paper used ranger defaults)
    Reports: AUC, accuracy, sensitivity, specificity, MCC
    """
    print(f"\n  Evaluating: {name} ({X.shape[1]} features)")

    outer_cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    rf = RandomForestClassifier(**RF_PARAMS)

    all_y_true  = []
    all_y_prob  = []
    all_y_pred  = []
    fold_aucs   = []

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Replace NaN/inf and clip extreme values before casting to float32
        X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
        X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0)
        X_train = np.clip(X_train, -1e15, 1e15)
        X_test  = np.clip(X_test,  -1e15, 1e15)

        rf.fit(X_train, y_train)
        y_prob = rf.predict_proba(X_test)[:, 1]
        y_pred = rf.predict(X_test)

        all_y_true.extend(y_test)
        all_y_prob.extend(y_prob)
        all_y_pred.extend(y_pred)
        fold_aucs.append(roc_auc_score(y_test, y_prob))

    all_y_true = np.array(all_y_true)
    all_y_prob = np.array(all_y_prob)
    all_y_pred = np.array(all_y_pred)

    # Metrics
    auc  = roc_auc_score(all_y_true, all_y_prob)
    acc  = accuracy_score(all_y_true, all_y_pred)
    mcc  = matthews_corrcoef(all_y_true, all_y_pred)
    tn, fp, fn, tp = confusion_matrix(all_y_true, all_y_pred).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    results = {
        "feature_set":   name,
        "auc":           round(auc, 4),
        "accuracy":      round(acc, 4),
        "sensitivity":   round(sens, 4),
        "specificity":   round(spec, 4),
        "mcc":           round(mcc, 4),
        "auc_std":       round(np.std(fold_aucs), 4),
        "fold_aucs":     fold_aucs,
        "y_true":        all_y_true,
        "y_prob":        all_y_prob,
    }

    print(f"    AUC:         {auc:.4f} ± {np.std(fold_aucs):.4f}")
    print(f"    Accuracy:    {acc:.4f}")
    print(f"    Sensitivity: {sens:.4f}")
    print(f"    Specificity: {spec:.4f}")
    print(f"    MCC:         {mcc:.4f}")

    return results


# ── Step 3: Feature importance ────────────────────────────────────────────────
def get_feature_importance(X: np.ndarray, y: np.ndarray,
                            feature_names: list) -> pd.DataFrame:
    """Train RF on full dataset and extract feature importances."""
    X_clean = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X_clean = np.clip(X_clean, -1e15, 1e15)
    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_clean, y)

    importance_df = pd.DataFrame({
        "feature":    feature_names,
        "importance": rf.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    return importance_df


# ── Step 4: Boruta feature selection ─────────────────────────────────────────
def boruta_selection(X: np.ndarray, y: np.ndarray,
                     feature_names: list, max_iter: int = 100) -> list:
    """
    Simplified Boruta: compare each feature's importance to the max importance
    of shuffled (shadow) features across multiple RF iterations.
    Features that consistently beat the shadow max are 'confirmed'.
    """
    try:
        from boruta import BorutaPy
        X_clean = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        X_clean = np.clip(X_clean, -1e15, 1e15)
        rf = RandomForestClassifier(n_estimators=100, max_features="sqrt",
                                    class_weight="balanced", random_state=RANDOM_STATE,
                                    n_jobs=-1)
        boruta = BorutaPy(rf, n_estimators="auto", random_state=RANDOM_STATE,
                          max_iter=max_iter, verbose=0)
        boruta.fit(X_clean, y)
        selected = [feature_names[i] for i in range(len(feature_names))
                    if boruta.support_[i]]
        tentative = [feature_names[i] for i in range(len(feature_names))
                     if boruta.support_weak_[i]]
        print(f"    Confirmed: {len(selected)} | Tentative: {len(tentative)}")
        return selected, tentative
    except ImportError:
        print("    boruta not installed — skipping (pip install boruta)")
        return [], []


# ── Step 5: Plot ROC curves ───────────────────────────────────────────────────
def plot_roc_curves(all_results: dict):
    fig, ax = plt.subplots(figsize=(7, 6))

    colors = {"chemical": "#2196F3", "go": "#4CAF50", "combined": "#F44336"}
    labels = {"chemical": "Chemical descriptors",
              "go":       "GO term features",
              "combined": "Combined"}

    for name, res in all_results.items():
        fpr, tpr, _ = roc_curve(res["y_true"], res["y_prob"])
        ax.plot(fpr, tpr,
                color=colors[name],
                label=f"{labels[name]} (AUC = {res['auc']:.3f})",
                lw=2)

    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title("ROC Curves — Nested 10-fold CV", fontsize=13)
    ax.legend(loc="lower right", fontsize=10)
    ax.grid(True, alpha=0.3)

    # Add paper's best AUC for reference
    ax.axhline(y=0, color="none")
    ax.text(0.55, 0.12, "Paper best AUC = 0.800 (combined)",
            fontsize=8, color="grey", style="italic")

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "roc_curves.png", dpi=150)
    plt.close()
    print(f"  Saved: plots/roc_curves.png")


def plot_feature_importance(importance_df: pd.DataFrame, top_n: int = 30):
    top = importance_df.head(top_n)
    fig, ax = plt.subplots(figsize=(8, 9))

    colors = ["#F44336" if not any(
        go in f for go in ["biosynthetic", "metabolic", "signaling",
                           "regulation", "response", "binding", "activity"]
    ) else "#4CAF50" for f in top["feature"]]

    ax.barh(top["feature"][::-1], top["importance"][::-1], color=colors[::-1])
    ax.set_xlabel("Mean Decrease in Impurity", fontsize=11)
    ax.set_title(f"Top {top_n} Feature Importances (Combined Model)", fontsize=12)
    ax.tick_params(axis="y", labelsize=8)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "feature_importance.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: plots/feature_importance.png")


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("="*55)
    print("PHASE 3: RANDOM FOREST — NESTED 10-FOLD CV")
    print("="*55)

    # Load features
    print("\nLoading feature sets...")
    feature_sets, merged = load_features()

    # Evaluate all three feature sets
    print("\n--- Nested CV Evaluation ---")
    all_results = {}
    for fname, (X, feat_names, y, names) in feature_sets.items():
        all_results[fname] = evaluate_feature_set(X, y, fname)

    # Summary table
    print("\n" + "="*55)
    print("RESULTS SUMMARY")
    print("="*55)
    print(f"{'Feature Set':<12} {'AUC':>8} {'±':>6} {'Acc':>8} {'Sens':>8} {'Spec':>8} {'MCC':>8}")
    print("-"*55)
    for name, res in all_results.items():
        print(f"{name:<12} {res['auc']:>8.4f} {res['auc_std']:>6.4f} "
              f"{res['accuracy']:>8.4f} {res['sensitivity']:>8.4f} "
              f"{res['specificity']:>8.4f} {res['mcc']:>8.4f}")
    print("-"*55)
    print(f"{'Paper best':<12} {'0.800':>8} {'':>6} {'':>8} {'':>8} {'':>8} {'':>8}")
    print("="*55)

    # Save results
    results_df = pd.DataFrame([
        {k: v for k, v in r.items()
         if k not in ("fold_aucs", "y_true", "y_prob")}
        for r in all_results.values()
    ])
    results_df.to_csv(OUTPUT_DIR / "results_summary.csv", index=False)

    # Feature importance on combined model
    print("\n--- Feature Importance (Combined Model) ---")
    X_combined, feat_names, y, _ = feature_sets["combined"]
    importance_df = get_feature_importance(X_combined, y, feat_names)
    importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
    print(f"  Top 10 features:")
    print(importance_df.head(10)[["feature", "importance"]].to_string(index=False))

    # Boruta feature selection on combined model
    print("\n--- Boruta Feature Selection (Combined Model) ---")
    selected, tentative = boruta_selection(X_combined, y, feat_names)
    if selected:
        boruta_df = pd.DataFrame({
            "feature": selected + tentative,
            "status":  ["confirmed"] * len(selected) + ["tentative"] * len(tentative)
        })
        boruta_df.to_csv(OUTPUT_DIR / "boruta_selected_features.csv", index=False)
        print(f"  Saved: data/boruta_selected_features.csv")

    # Plots
    print("\n--- Generating Plots ---")
    plot_roc_curves(all_results)
    plot_feature_importance(importance_df)

    print(f"\n✅ Phase 3 complete!")
    print(f"   Best AUC: {max(r['auc'] for r in all_results.values()):.4f} "
          f"({max(all_results, key=lambda k: all_results[k]['auc'])} features)")
    print(f"   Paper:    0.800 (combined features)")

PHASE 3: RANDOM FOREST — NESTED 10-FOLD CV

Loading feature sets...
Chemical descriptors: 179 features, 2521 compounds
GO term features:     380 features, 2521 compounds
After merge:          2521 compounds
  Positive: 428 | Negative: 2093

--- Nested CV Evaluation ---

  Evaluating: chemical (179 features)
    AUC:         0.8096 ± 0.0365
    Accuracy:    0.8520
    Sensitivity: 0.2874
    Specificity: 0.9675
    MCC:         0.3616

  Evaluating: go (380 features)
    AUC:         0.7110 ± 0.0271
    Accuracy:    0.6129
    Sensitivity: 0.8575
    Specificity: 0.5628
    MCC:         0.3156

  Evaluating: combined (559 features)
    AUC:         0.8365 ± 0.0371
    Accuracy:    0.8651
    Sensitivity: 0.3318
    Specificity: 0.9742
    MCC:         0.4290

RESULTS SUMMARY
Feature Set       AUC      ±      Acc     Sens     Spec      MCC
-------------------------------------------------------
chemical       0.8096 0.0365   0.8520   0.2874   0.9675   0.3616
go             0.7110 0.0271 